## Federated Learning Demo

### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: April 7, 2026

---

### OBJECTIVES: 

- Understand the concept of federated learning through a computational example
- Train a model across multiple clients without sharing their raw data

### TO DO:
- Review and run all of the code carefully
- Notice how the global model and local models update each other
- Complete the next steps at the bottom of this notebook

---

NOTE: Make sure the notebook is using this kernel: `PyTorch 2.7.0`

In [1]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset

#### Create the dataset

In [15]:
# Simple linear data: y = 2*x + 1
x = torch.linspace(0, 9, 100).view(-1, 1)
y = 2 * x + 1

# Split dataset into 3 clients
# A TensorDataset binds features and labels together into a single object
client1_data = TensorDataset(x[:33], y[:33])
client2_data = TensorDataset(x[33:66], y[33:66])
client3_data = TensorDataset(x[66:], y[66:])

clients = [
    DataLoader(client1_data, batch_size=10, shuffle=True),
    DataLoader(client2_data, batch_size=10, shuffle=True),
    DataLoader(client3_data, batch_size=10, shuffle=True)
]

#### Inspect some of the data

In [14]:
print(f'length of x: {len(x)}')
print(f'first few observations of x: {x[:5]}')
print(f'first few observations of y: {y[:5]}')

length of x: 100
first few observations of x: tensor([[0.0000],
        [0.0909],
        [0.1818],
        [0.2727],
        [0.3636]])
first few observations of y: tensor([[1.0000],
        [1.1818],
        [1.3636],
        [1.5455],
        [1.7273]])


#### Define a simple linear model at the global level
The global model will be shared by the clients

In [49]:
class SimpleLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1, 1)
    def forward(self, x):
        return self.fc(x)

global_model = SimpleLinear()

#### Define Local Training Function
The local models are used by the clients

In [50]:
def train_local(model, dataloader, epochs, lr):
    optimizer = optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    model.train()

    for _ in range(epochs):
        for xt, yt in dataloader:
            optimizer.zero_grad()
            pred = model(xt)
            loss = loss_fn(pred, yt)
            loss.backward()
            optimizer.step()
    return model.state_dict()

#### Federated Training Loop using FedAvg

In [56]:
rounds = 200   # each round consists of updates across the clients
lr = 0.01    # learning rate for model
epochs = 5 

for r in range(rounds):
    print('')
    print(f'Round: {r+1}')
    print(f'global_model: {global_model.state_dict()}')
    print('')

    local_weights = []

    # Each client trains locally
    i = 0
    for client_loader in clients:
        local_model = SimpleLinear()
        local_model.load_state_dict(global_model.state_dict())  # initialize with global model
        local_weights.append(train_local(local_model, client_loader, epochs, lr))
        print(f'client_loader: {i+1}, local_weights: {local_weights} \n')
        i += 1

    # FedAvg: average weights across clients for each parameter
    new_weights = {}
    for key in local_weights[1]:
        print('--------------------------')
        print(f'key: {key}')
        new_weights[key] = sum(w[key] for w in local_weights) / len(local_weights)
        print(f'new_weights: {new_weights[key]}')
    print('--------------------------')
    
    global_model.load_state_dict(new_weights)
    print(f"Round {r+1} completed.")


Round: 1
global_model: OrderedDict({'fc.weight': tensor([[2.0283]]), 'fc.bias': tensor([0.8902])})

client_loader: 1, local_weights: [OrderedDict({'fc.weight': tensor([[2.0444]]), 'fc.bias': tensor([0.9084])})] 

client_loader: 2, local_weights: [OrderedDict({'fc.weight': tensor([[2.0444]]), 'fc.bias': tensor([0.9084])}), OrderedDict({'fc.weight': tensor([[2.0239]]), 'fc.bias': tensor([0.8909])})] 

client_loader: 3, local_weights: [OrderedDict({'fc.weight': tensor([[2.0444]]), 'fc.bias': tensor([0.9084])}), OrderedDict({'fc.weight': tensor([[2.0239]]), 'fc.bias': tensor([0.8909])}), OrderedDict({'fc.weight': tensor([[2.0146]]), 'fc.bias': tensor([0.8890])})] 

--------------------------
key: fc.weight
new_weights: tensor([[2.0276]])
--------------------------
key: fc.bias
new_weights: tensor([0.8961])
--------------------------
Round 1 completed.

Round: 2
global_model: OrderedDict({'fc.weight': tensor([[2.0276]]), 'fc.bias': tensor([0.8961])})

client_loader: 1, local_weights: [Orde

#### Test the Global Model

In [57]:
global_model.eval()
with torch.no_grad():
    test_point = 5.0
    test_input = torch.tensor([[test_point]])
    print(f"Prediction for x={test_point}:", global_model(test_input).item())

Prediction for x=5.0: 10.999999046325684


#### Conclusions

- Notice that clients train locally; there is no data sharing
- Global model weights are updated using the FedAvg algorithm

#### Next Steps

Trying increasing the rounds / epochs for better accuracy